# Study 9: does the linking function memorize the data? ZOIB vs original

Two tests of whether the per-feature linking function fits the ratings through memorization rather than
through the GP's generalization, for the **original** free Beta-mixture linking (`results/study9-fit/`)
and the **per-feature ZOIB** linking (`results/study9-fit-zoib/`), each with a fitted and a flat
(distance-blind) length scale.

1. **Leave-one-condition-out (LOCO) cross-validation.** For each condition in turn, the linking is refit on
   the other three conditions' participants, and the held-out condition's ratings are predicted from its
   training statements alone (ratings-free coherence). Memorizing each feature's ratings can only predict
   the held-out condition at the features' pooled level; predicting how the held-out condition *shifts*
   away from that requires the coherence field. Held-out participants within a condition can't test this,
   because each feature's distribution is stable across people.
2. **Parameter-penalized comparison (BIC / AIC)** on a common discrete scale, charging each linking for its
   parameters: 4 per feature for the original, 8 per feature for the ZOIB.

**Common scale.** Every model gives a probability to each of the 101 slider values (they sum to 1), so
log-likelihoods are comparable across linkings; a coarse version with 12 bins (`{0}`, `1–9`, …, `90–99`,
`{100}`) removes the effect of round-number heaping. Same scoring as section 5 of
`study9-gp-vs-flat-zoib.ipynb`.

**θ.** By default each model keeps its full-data posterior-median θ in every fold. That θ was fit with the
held-out condition's ratings included, a mild leak that favors all GP models equally. Set `REFIT_THETA = True`
(below) to refit θ with VBMC on the three training conditions in every fold instead: 4 folds × 4 models = 16
VBMC fits, roughly 1.5–2 h, cached to `results/study9-loco/` so each is done once.

In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax.numpy as jnp
from scipy.special import betainc

import inference as inf
import linking as lk
import plot_style
plot_style.use_lexend()          # Lexend everywhere (DejaVu Sans fills in σ/β)

REFIT_THETA = False              # True: refit θ by VBMC without the held-out condition (slow, cached)
LOCO_DIR = os.path.join('results', 'study9-loco')
MAX_EVALS_LOCO = 400
SEED = 37

DATA_CSV = '../../data/study9_prereg.csv'
geom = inf.load_geometry()
J, CONDS = geom['J'], list(inf.CONDITIONS)
features = geom['test_feature_names']
raw, _ = inf.load_responses(DATA_CSV, clip_interior=False)      # ZOIB fits on raw ratings
nudged, _ = inf.load_responses(DATA_CSV)                        # the original linking fits on nudged ones
N_cond = {c: int(raw[c].shape[0]) for c in CONDS}
N_RATINGS = sum(N_cond.values()) * J
emp = {c: np.asarray(raw[c]).mean(0) for c in CONDS}

def med(path):
    z = np.load(path, allow_pickle=True)
    return tuple(float(np.median(z[k])) for k in ('ls_samples', 'mu0_samples', 'sigma_samples', 'beta_samples'))

MODELS = {
    'ZOIB, fitted': {'link': 'zoib', 'post': 'results/study9-fit-zoib/posterior.npz', 'ls_fixed': None},
    'ZOIB, flat':   {'link': 'zoib', 'post': 'results/study9-fit-zoib/posterior-null.npz', 'ls_fixed': inf.LS_FLAT},
    'free, fitted': {'link': 'free', 'post': 'results/study9-fit/fitted-feature-shapes.npz', 'ls_fixed': None},
    'free, flat':   {'link': 'free', 'post': 'results/study9-fit/fitted-feature-shapes-null.npz', 'ls_fixed': inf.LS_FLAT},
}
N_THETA = {'fitted': 4, 'flat': 3}
N_LINK = {'free': 4 * J, 'zoib': 8 * J}
for n, m in MODELS.items():
    m['theta'] = med(m['post'])
    ls, mu0, sig, b = m['theta']
    print(f"{n:<13} ls={ls:<7.3g} mu_0={mu0:+.3f} sigma={sig:.3f} beta={b:.3f}")

COLORS = {'ZOIB, fitted': '#2a78d6', 'ZOIB, flat': '#9ec5f0', 'free, fitted': '#eb6834', 'free, flat': '#f5b899',
          'memorize: feature histogram': '#8a8984'}
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})

## Scoring helpers (same as `study9-gp-vs-flat-zoib.ipynb` section 5)

In [ ]:
EDGES = np.concatenate([[0.0], (np.arange(100) + 0.5) / 100, [1.0]])     # 102 edges -> 101 slider values
GROUPS = [np.array([0])] + [np.arange(max(1, lo), lo + 10) for lo in range(0, 100, 10)] + [np.array([100])]
GROUP_OF = np.empty(101, int)
for g, vals in enumerate(GROUPS):
    GROUP_OF[vals] = g

def beta_bin_mass(a, b):
    F = betainc(np.asarray(a)[:, None], np.asarray(b)[:, None], EDGES[None, :])
    return np.diff(F, axis=1)                                            # (J, 101)

def probs_free(shapes, pz1_c):
    w = np.asarray(pz1_c)[:, None]
    return w * beta_bin_mass(shapes[:, 0], shapes[:, 1]) + (1 - w) * beta_bin_mass(shapes[:, 2], shapes[:, 3])

def probs_zoib(link_theta, pz1_c):
    q = lk.params_to_numpy(lk.unpack(jnp.asarray(link_theta)))
    comp = []
    for cl in ('kl', 'nkl'):
        P = (1 - q['pi0_' + cl] - q['pi1_' + cl])[:, None] * beta_bin_mass(q['a_' + cl], q['b_' + cl])
        P[:, 0] += q['pi0_' + cl]; P[:, 100] += q['pi1_' + cl]
        comp.append(P)
    w = np.asarray(pz1_c)[:, None]
    return w * comp[0] + (1 - w) * comp[1]

def hist_probs(R, alpha=0.5):
    k = np.round(np.asarray(R) * 100).astype(int)
    C = np.stack([np.bincount(k[:, j], minlength=101) for j in range(k.shape[1])]).astype(float) + alpha
    return C / C.sum(1, keepdims=True)

def hist_probs_coarse(R, alpha=0.5):
    k = GROUP_OF[np.round(np.asarray(R) * 100).astype(int)]
    C = np.stack([np.bincount(k[:, j], minlength=len(GROUPS)) for j in range(k.shape[1])]).astype(float) + alpha
    C /= C.sum(1, keepdims=True)
    P = np.zeros((C.shape[0], 101))
    for g, vals in enumerate(GROUPS):
        P[:, vals] = C[:, [g]] / len(vals)
    return P

def score(P, R, scale='fine'):
    k = np.round(np.asarray(R) * 100).astype(int)
    if scale == 'coarse':
        P = np.stack([P[:, vals].sum(1) for vals in GROUPS], axis=1); k = GROUP_OF[k]
    return float(np.sum(np.log(np.maximum(P[np.arange(P.shape[0])[None, :], k], 1e-300))))

def mean_of(P):
    return P @ (np.arange(101) / 100)                                    # (J,) expected rating

def pz1_at(theta):
    ls, mu0, sig, b = theta
    return {c: np.asarray(v) for c, v in inf.prior_pz1_modes(geom, ls, mu0, sig, beta_speaker=b).items()}

def fit_link(m, pz1, train):
    # fit the model's linking on the training conditions; return a function pz1_c -> (J, 101) probabilities
    if m['link'] == 'zoib':
        th, _ = lk.fit({c: raw[c] for c in train}, {c: pz1[c] for c in train})
        return lambda pz1_c: probs_zoib(th, pz1_c)
    shapes = inf.fit_link_shapes({c: nudged[c] for c in train}, {c: pz1[c] for c in train})
    return lambda pz1_c: probs_free(shapes, pz1_c)

# sanity: probabilities sum to 1 for both linkings
_pz = pz1_at(MODELS['ZOIB, fitted']['theta'])
assert np.allclose(fit_link(MODELS['ZOIB, fitted'], _pz, CONDS)(_pz[CONDS[0]]).sum(1), 1)
assert np.allclose(fit_link(MODELS['free, fitted'], _pz, CONDS)(_pz[CONDS[0]]).sum(1), 1)
print('value probabilities sum to 1 for both linkings')

## 1. Leave-one-condition-out cross-validation

For each held-out condition: θ (full-data, or refit without the held-out condition if `REFIT_THETA`), then
the linking fit on the other three conditions, then the held-out condition's ratings scored from its
ratings-free coherence. The baseline memorizes each feature's rating histogram from the other three
conditions (it predicts the same distribution for the held-out condition). Also recorded: each model's
predicted mean rating per feature for the held-out condition, to compare **shifts** below.

In [ ]:
def theta_for_fold(name, m, c_out):
    if not REFIT_THETA:
        return m['theta']
    os.makedirs(LOCO_DIR, exist_ok=True)
    path = os.path.join(LOCO_DIR, f"{name.replace(', ', '-')}_without-{c_out}.npz")
    if not os.path.exists(path):
        train = [c for c in CONDS if c != c_out]
        resp = {c: (raw if m['link'] == 'zoib' else nudged)[c] for c in train}
        print(f"  VBMC: {name} without {c_out} ...")
        fit = inf.run_vbmc(geom, resp, link=m['link'], ls_fixed=m['ls_fixed'],
                           max_evals=MAX_EVALS_LOCO, seed=SEED, verbose=False)
        np.savez_compressed(path, **{k: fit[k] for k in ('ls_samples', 'mu0_samples', 'sigma_samples', 'beta_samples')},
                            meta=json.dumps({k: fit[k] for k in ('elbo', 'func_count', 'convergence_status', 'runtime_s')}))
    return med(path)

loco_rows, loco_means = [], {}
for c_out in CONDS:
    train = [c for c in CONDS if c != c_out]
    R_out = np.asarray(raw[c_out])
    for name, m in MODELS.items():
        theta = theta_for_fold(name, m, c_out)
        pz1 = pz1_at(theta)
        predict = fit_link(m, pz1, train)
        P_out = predict(pz1[c_out])
        loco_means[name, c_out] = mean_of(P_out)
        loco_means[name, 'train', c_out] = np.mean([mean_of(predict(pz1[c])) for c in train], axis=0)
        loco_rows.append({'model': name, 'held out': c_out,
                          'fine': score(P_out, R_out), 'coarse': score(P_out, R_out, 'coarse')})
    R_tr = np.concatenate([np.asarray(raw[c]) for c in train])
    H = hist_probs(R_tr); Hc = hist_probs_coarse(R_tr)
    loco_means['memorize: feature histogram', c_out] = mean_of(H)
    loco_means['memorize: feature histogram', 'train', c_out] = mean_of(H)
    loco_rows.append({'model': 'memorize: feature histogram', 'held out': c_out,
                      'fine': score(H, R_out), 'coarse': score(Hc, R_out, 'coarse')})
    print(f"held out {c_out}: done")

loco = pd.DataFrame(loco_rows)
order = list(MODELS) + ['memorize: feature histogram']
tot = loco.groupby('model')[['fine', 'coarse']].sum().loc[order]
tot['fine vs histogram'] = tot['fine'] - tot.loc['memorize: feature histogram', 'fine']
tot['coarse vs histogram'] = tot['coarse'] - tot.loc['memorize: feature histogram', 'coarse']
tot['coarse per rating'] = tot['coarse'] / N_RATINGS
print(f"theta per fold: {'refit by VBMC without the held-out condition' if REFIT_THETA else 'full-data posterior median'}")
display(loco.pivot(index='model', columns='held out', values='coarse').loc[order, CONDS].round(1))
tot.round(2)

### 1b. Predicting the held-out condition's *shift*

For each held-out condition and feature: the observed shift is the held-out condition's mean rating minus that
feature's mean over the three training conditions; the predicted shift is the model's predicted mean for the
held-out condition minus its predicted mean over the training conditions. Memorization predicts a shift of 0
everywhere. Pooled over the 4 folds × 15 features = 60 points.

In [ ]:
shift_rows = []
obs_shift = {c: emp[c] - np.mean([emp[t] for t in CONDS if t != c], axis=0) for c in CONDS}
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 4.2), sharex=True, sharey=True)
lim = 1.15 * max(np.abs(np.concatenate(list(obs_shift.values()))).max(),
                 max(np.abs(loco_means[n, c] - loco_means[n, 'train', c]).max() for n in MODELS for c in CONDS))
for ax, name in zip(axes, MODELS):
    pred = np.concatenate([loco_means[name, c] - loco_means[name, 'train', c] for c in CONDS])
    obs = np.concatenate([obs_shift[c] for c in CONDS])
    r = np.corrcoef(pred, obs)[0, 1] if np.ptp(pred) > 1e-9 else np.nan
    sse, sst = np.sum((obs - pred) ** 2), np.sum(obs ** 2)
    shift_rows.append({'model': name, 'r(pred shift, obs shift)': r, 'R2 vs zero shift': 1 - sse / sst,
                       'mean |pred shift|': np.mean(np.abs(pred)), 'mean |obs shift|': np.mean(np.abs(obs))})
    ax.plot(pred, obs, 'o', color=COLORS[name], ms=6, mec='white')
    ax.plot([-lim, lim], [-lim, lim], color='#8a8984', lw=1, ls='--')
    ax.axhline(0, color='#e6e5e1', lw=1); ax.axvline(0, color='#e6e5e1', lw=1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_title(f"{name}\nr = {r:+.2f}", fontsize=10); ax.set_xlabel('predicted shift')
axes[0].set_ylabel('observed shift (held-out − other conditions)')
fig.suptitle('Leave-one-condition-out: predicted vs observed shift of each feature in the held-out condition', fontsize=12)
fig.tight_layout(); plt.show()
shift_rows.append({'model': 'memorize: feature histogram', 'r(pred shift, obs shift)': np.nan, 'R2 vs zero shift': 0.0,
                   'mean |pred shift|': 0.0, 'mean |obs shift|': shift_rows[0]['mean |obs shift|']})
pd.DataFrame(shift_rows).set_index('model').round(3)

## 2. Parameter-penalized comparison (BIC / AIC)

In-sample log-likelihood on the common discrete scale (linking fit to all participants at each model's
full-data θ), penalized for the number of fitted parameters k = linking parameters (4·J for the original,
8·J for the ZOIB) + θ parameters (4 fitted, 3 flat). N = number of ratings. Lower BIC / AIC = better.
For reference, the histograms are charged one parameter per slider value (fine: 100·J; coarse: 11·J).

Caveats: the linking parameters are profiled (fit per θ), not integrated, so this is an approximation; the
ZOIB's weak prior on the point masses isn't counted; and BIC's per-parameter charge (ln N ≈ 8.7) is steep
for 60 extra parameters, so read it together with the LOCO results above.

In [ ]:
rows = []
for name, m in MODELS.items():
    pz1 = pz1_at(m['theta'])
    predict = fit_link(m, pz1, CONDS)
    ll_f = sum(score(predict(pz1[c]), raw[c]) for c in CONDS)
    ll_c = sum(score(predict(pz1[c]), raw[c], 'coarse') for c in CONDS)
    k = N_LINK[m['link']] + N_THETA['flat' if m['ls_fixed'] is not None else 'fitted']
    rows.append({'model': name, 'k': k, 'logL fine': ll_f, 'logL coarse': ll_c})
R_all = np.concatenate([np.asarray(raw[c]) for c in CONDS])
rows.append({'model': 'memorize: feature histogram', 'k': 100 * J,
             'logL fine': score(hist_probs(R_all), R_all), 'logL coarse': np.nan})
rows.append({'model': 'memorize: feature histogram (coarse)', 'k': 11 * J,
             'logL fine': np.nan, 'logL coarse': score(hist_probs_coarse(R_all), R_all, 'coarse')})
ic = pd.DataFrame(rows).set_index('model')
for s in ('fine', 'coarse'):
    ic[f'BIC {s}'] = ic['k'] * np.log(N_RATINGS) - 2 * ic[f'logL {s}']
    ic[f'AIC {s}'] = 2 * ic['k'] - 2 * ic[f'logL {s}']
ic.round(1)

## How to read the results

- **LOCO totals (section 1):** the held-out condition's ratings under each model. Beating the feature histogram
  here (positive "vs histogram") means the model predicts an unseen condition better than memorizing each
  feature; fitted beating flat within a linking means the distance-based field helps predict a new condition.
- **Shift (1b):** the sharpest test of memorization. A linking that only memorizes feature levels predicts no
  shift; any positive r / R² comes from the coherence field's condition-specific predictions.
- **BIC / AIC (section 2):** whether the ZOIB's extra 60 parameters are worth it in-sample. LOCO is the better
  guide when the two disagree, since it measures prediction of new data directly.